# Day 028 Project Solution — SpreadsheetAnalyst

A `SpreadsheetAnalyst` that reads tabular data, generates AI insights, and writes a formatted two-sheet report.

In [ ]:
import json, os, tempfile
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font
import ollama


def read_sheet_rows(path: str, sheet_name: str | None = None) -> list[dict]:
    wb = load_workbook(path)
    ws = wb[sheet_name] if sheet_name else wb.active
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return []
    headers = [str(h) for h in rows[0]]
    return [dict(zip(headers, row)) for row in rows[1:]]


def write_sheet_rows(
    path: str,
    sheet_name: str,
    headers: list[str],
    rows: list[dict],
) -> None:
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name
    ws.append(headers)
    for row in rows:
        ws.append([row.get(h) for h in headers])
    wb.save(path)


def append_sheet_row(path: str, sheet_name: str, row: dict) -> None:
    wb = load_workbook(path)
    ws = wb[sheet_name]
    headers = [
        ws.cell(row=1, column=c).value
        for c in range(1, ws.max_column + 1)
    ]
    ws.append([row.get(h) for h in headers])
    wb.save(path)


def bold_header_row(path: str, sheet_name: str) -> None:
    wb = load_workbook(path)
    ws = wb[sheet_name]
    bold_font = Font(bold=True)
    for cell in ws[1]:
        cell.font = bold_font
    wb.save(path)


def ai_analyze_sheet(path: str, question: str, model: str = "llama3.2") -> str:
    rows = read_sheet_rows(path)
    data_str = json.dumps(rows[:20], indent=2)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data analyst. Answer questions about tabular data "
                    "concisely and precisely."
                ),
            },
            {
                "role": "user",
                "content": f"Data:\n{data_str[:3000]}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]


class SpreadsheetAnalyst:
    def __init__(self, model: str = "llama3.2"):
        self.model = model

    def analyze(self, rows: list[dict], question: str) -> str:
        data_str = json.dumps(rows[:20], indent=2)
        response = ollama.chat(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": "You are a data analyst. Answer concisely.",
                },
                {
                    "role": "user",
                    "content": f"Data:\n{data_str[:3000]}\n\nQuestion: {question}",
                },
            ],
        )
        return response["message"]["content"]

    def write_summary(
        self, input_path: str, analysis: str, output_path: str
    ) -> None:
        rows = read_sheet_rows(input_path)
        wb = Workbook()
        ws_data = wb.active
        ws_data.title = "Data"
        if rows:
            headers = list(rows[0].keys())
            ws_data.append(headers)
            for row in rows:
                ws_data.append([row.get(h) for h in headers])
            for cell in ws_data[1]:
                cell.font = Font(bold=True)
        ws_sum = wb.create_sheet("Summary")
        ws_sum.append(["AI Analysis"])
        ws_sum.append([analysis])
        wb.save(output_path)

    def run(
        self, input_path: str, question: str, output_path: str
    ) -> dict:
        rows = read_sheet_rows(input_path)
        analysis = self.analyze(rows, question)
        self.write_summary(input_path, analysis, output_path)
        return {
            "rows_analyzed": len(rows),
            "analysis": analysis,
            "output": output_path,
        }

## Action 1 — Create Sample XLSX with Bold Headers

In [ ]:
SAMPLE_DATA = [
    {'Product': 'Alpha Pro',  'Q1': 1200, 'Q2': 1350, 'Q3':  980, 'Q4': 1600},
    {'Product': 'Beta Max',   'Q1':  870, 'Q2':  920, 'Q3': 1100, 'Q4': 1250},
    {'Product': 'Gamma Plus', 'Q1': 2100, 'Q2': 1980, 'Q3': 2200, 'Q4': 2450},
    {'Product': 'Delta Lite', 'Q1':  450, 'Q2':  510, 'Q3':  390, 'Q4':  620},
    {'Product': 'Epsilon X',  'Q1': 3200, 'Q2': 3100, 'Q3': 3450, 'Q4': 3600},
]
HEADERS = ['Product', 'Q1', 'Q2', 'Q3', 'Q4']

input_path = os.path.join(tempfile.gettempdir(), 'day028_input.xlsx')
write_sheet_rows(input_path, 'Sales', HEADERS, SAMPLE_DATA)
bold_header_row(input_path, 'Sales')

rows = read_sheet_rows(input_path, 'Sales')
print(f'Created input XLSX with {len(rows)} data rows')
print(f'Columns: {list(rows[0].keys())}')
print(f'First row: {rows[0]}')

## Action 2 — Analyze Data with AI

In [ ]:
QUESTION = (
    'Which product has the highest total annual sales? '
    'What growth trends do you observe across the quarters?'
)

analyst = SpreadsheetAnalyst()
analysis = analyst.analyze(rows, QUESTION)
print('AI Analysis:')
print(analysis)

## Action 3 — Write Formatted Report and Verify

In [ ]:
output_path = os.path.join(tempfile.gettempdir(), 'day028_report.xlsx')
analyst.write_summary(input_path, analysis, output_path)

# Verify the output
wb = load_workbook(output_path)
print(f'Output sheets: {wb.sheetnames}')

ws_data = wb['Data']
print(f'Data sheet rows: {ws_data.max_row} (including header)')
print(f'Header row bold: {ws_data["A1"].font.bold}')

ws_sum = wb['Summary']
summary_cell = ws_sum.cell(row=2, column=1).value or ''
print(f'Summary sheet preview: {summary_cell[:100]}')

print('\nAnalysis complete!')